# Probe: MoE-head-only router/adaptor DQA-MoX

- created_utc: 2026-05-11T15:52:41+00:00
- target_mAP50: 0.600
- workspace: `/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27g_probe_moe_head_only_router_r1`
- log: `/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27g_probe_moe_head_only_router_r1/logs/27g_probe_moe_head_only_router_r1_train.log`
- research_note: `/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/reports/27_research_note_iter_000_27g_probe_moe_head_only_router_r1.md`

## Current Results

| trial | best mAP50 | mAP50:95 |
|---|---:|---:|
| 27a_soft_mixture_head_first_40_10 |  |  |
| 27b_localization_uncertainty_strict_then_open |  |  |
| 27b_probe_localization_uncertainty_r2 | 0.462000 | 0.260000 |
| 27c_probe_k6_night_tail_r2 | 0.459000 | 0.250000 |
| 27d_probe_teacher_residual_mixpl_r2 | 0.462000 | 0.259000 |
| 27e_probe_clean_day_expert_anchor_r2 | 0.462000 | 0.259000 |

## Hypothesis

27fでposthoc expert graftは0.462の壁を破れず、MoE residualを強くするとmAP50:95が崩れた。つまりexpert slot自体より、router/adaptorをbase detectorから分離して訓練時に作る必要がある。ここではbackbone/neck/shared headを固定し、head.routerとhead.expert_mだけを更新するmoe_head scopeを追加し、sourceを多めにしてpseudoGTをdomain/router信号として使う1-round probeにする。

## Paper Basis

- FedMoX/PSSFL: https://arxiv.org/abs/2508.16568
  FedMoX treats the practical setting as server labeled high-resolution data plus client unlabeled low-resolution data, and uses sparse MoE with a spatial router and Soft-Mixture to stabilize semi-supervised FL.
- FedDG-MoE: https://openaccess.thecvf.com/content/CVPR2025W/FedVision/papers/Radwan_FedDG-MoE_Test-Time_Mixture-of-Experts_Fusion_for_Federated_Domain_Generalization_CVPRW_2025_paper.pdf
  FedDG-MoE stores client-specific MoE adapters and fuses them using domain similarity at test time. For our YOLO latent MoE, this argues against repeatedly averaging all expert residuals into one bland head; the next aggressive probe should preserve client residual experts as separate slots in a single checkpoint.
- Uncertainty-aware Long-tailed Weights: https://arxiv.org/abs/2503.09974
  This work points out that confidence thresholds are brittle under over-confidence and long-tail scarcity. DQA should downweight uncertain/tail pseudo labels rather than only drop them, which supports residual expert grafting with small or blended tail adapters.
- TMLR 2025 SSOD Building Blocks: https://openreview.net/forum?id=vRYt8QLKqK
  The paper analyzes real-world SSOD failures from class imbalance, label noise, and missing detections. This matches our observation that adding pseudo boxes improves source-val a little but hurts some night splits unless local experts are preserved.


In [ ]:
import csv
import json
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path('/app/Object_Detection')
WORKSPACE = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27g_probe_moe_head_only_router_r1')
LOG_PATH = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27g_probe_moe_head_only_router_r1/logs/27g_probe_moe_head_only_router_r1_train.log')
METRICS_PATH = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27g_probe_moe_head_only_router_r1/stats/18_client_balanced_single_injection_dqamox_final_metrics.csv')
CMD = ['/opt/venv/bin/python', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/scripts/run_scene_daynight_dqa_18_client_balanced_single_injection_dqamox.py', '--workspace-root', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27g_probe_moe_head_only_router_r1', '--repair-baseline-rounds', '0', '--source-workspace', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/08_full_latent_dqamox_from_warmup', '--source-repair-baseline-rounds', '30', '--target-map50', '0.6', '--num-experts', '4', '--top-k', '1', '--router-temperature', '0.8', '--router-balance-weight', '0.04', '--router-entropy-weight', '0.0', '--dqa-client-balance-stats', '--dqa-client-balance-target', 'median', '--dqa-client-balance-max-scale', '4.0', '--load-bias-strength', '0.25', '--batch-size', '80', '--workers', '8', '--gpus', '2', '--max-images-per-client', '0', '--master-port', '39000', '--evaluate', '--classwise', '--no-eval-plots', '--force', '--warmup-epochs', '50', '--client-limit', '1400', '--client-sampling-ratio', '1.000', '--client-sampling-seed', '270512', '--phase1-rounds', '1', '--phase2-rounds', '0', '--phase1-train-scope', 'moe_head', '--phase1-repair-train-scope', 'moe_head', '--phase1-client-epochs', '1', '--phase1-client-lr', '0.00042', '--phase1-source-repeat', '4', '--phase1-pseudo-repeat', '1', '--phase1-loss-box', '0.00008', '--phase2-train-scope', 'moe_head', '--phase2-repair-train-scope', 'moe_head', '--phase2-client-epochs', '1', '--phase2-client-lr', '0.00004', '--phase2-source-repeat', '3', '--phase2-pseudo-repeat', '1', '--phase2-loss-box', '0.00003', '--server-repair-epochs', '1', '--server-repair-lr', '0.00030', '--server-repair-loss-box', '0.0006', '--dqa-temperature', '0.75', '--dqa-uniform-mix', '0.10', '--dqa-classwise-blend', '0.30', '--dqa-stability-lambda', '0.50', '--dqa-server-anchor', '0.64', '--dqa-min-server-alpha', '0.58', '--dqa-residual-blend', '0.12', '--late-dqa-server-anchor', '0.58', '--late-dqa-min-server-alpha', '0.52', '--late-dqa-residual-blend', '0.10', '--curriculum-start-round', '2', '--expert-keep-fraction', '0.82', '--expert-max-class-fraction', '0.32', '--actual-max-class-fraction', '0.42', '--late-expert-keep-fraction', '0.90', '--late-expert-max-class-fraction', '0.38', '--late-actual-max-class-fraction', '0.48', '--min-score', '0.20', '--min-stability', '0.58', '--late-min-score', '0.16', '--late-min-stability', '0.48', '--max-boxes-per-image', '10', '--skip-warmup-training', '--warmup-checkpoint', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/08_full_latent_dqamox_from_warmup/checkpoints/round000_latent_dqamox_warmup.pt']

WORKSPACE.mkdir(parents=True, exist_ok=True)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
(WORKSPACE / "stats").mkdir(parents=True, exist_ok=True)
(WORKSPACE / "stats" / "27_notebook_command.json").write_text(
    json.dumps({"command": CMD}, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
print(" ".join(CMD))
with LOG_PATH.open("w", encoding="utf-8") as log:
    proc = subprocess.run(CMD, cwd=REPO_ROOT, stdout=log, stderr=subprocess.STDOUT, check=False)
print("returncode", proc.returncode)
print("log", LOG_PATH)
if proc.returncode != 0:
    raise SystemExit(proc.returncode)


In [ ]:
import csv
from pathlib import Path

METRICS_PATH = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27g_probe_moe_head_only_router_r1/stats/18_client_balanced_single_injection_dqamox_final_metrics.csv')
rows = list(csv.DictReader(METRICS_PATH.open(encoding="utf-8"))) if METRICS_PATH.exists() else []
for row in rows:
    print(row)
